<a href="https://colab.research.google.com/github/drishikaaneja/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*
**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I'm going with Lane 2 mainly for two reasons. First, the problem itself — a big content inventory, limited review capacity, and the question of which pages to look at first — is a ranking problem, and this dataset has the most signal for exactly that. Second, it connects to work I did in my last internship on time-series trend and anomaly detection. That experience becomes directly useful later when I upgrade the label from the current-window `trend_direction` proxy to a proper future-window one (features from the prior 90 days, outcome over the next 30 days), which is what the lane guide recommends anyway. Keeping this provisional till Week 4, but the numbers below make me reasonably confident.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Checking if this lane has enough signal to work with
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_URL)

print("rows:", len(df), "| clients:", df["client_id"].nunique())
print()
print("trend_direction distribution (the starter proxy label):")
print(df["trend_direction"].value_counts())


rows: 30000 | clients: 32

trend_direction distribution (the starter proxy label):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*
**Research question:** Given a client's content inventory, which pages should a reviewer look at first for refresh, expansion, protection, or pruning — ranked by evidence, with a reason code for each page?

**The decision it improves:** how a content team spends its limited weekly review capacity. Right now the alternatives are arbitrary order, gut feel, or one fixed rule.

**Who acts on it:** a content manager or SEO reviewer. They take the top-K pages from the queue, check the reason code (say, "declining with demand"), open the page, and decide — refresh, expand, prune, or just monitor. The system suggests the order; the human makes the actual call.

**Cost of a wrong call:** a false positive wastes a review slot on a healthy page, which is annoying but cheap. A false negative is worse — a genuinely decaying, high-demand page sits unreviewed while it keeps bleeding traffic that took months to build. Since misses cost more than false alarms, ranked-queue metrics like precision@K matter more here than plain accuracy.

**Putting the frame in one paragraph:**

> For a content manager reviewing a client's inventory, deciding which pages to review first each week, I will build a ranked review queue with reason codes from the starter dataset (and later the warehouse daily facts), scoring each page's review priority, measured by precision@K on held-out clients. A wrong call costs either a wasted review slot (false positive, cheap) or an unreviewed decaying high-demand page (false negative, expensive). A plain rule isn't enough because the signal is spread across several interacting fields — trend, volume, position, engagement, freshness — and the starter results already show a learned ranking beating the fixed rule (Precision@50: 0.74 vs 0.24). I will claim only observed, directional, decision-support results.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Scale of the problem vs realistic review capacity — why the ORDER is the decision
candidates = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]

weekly_capacity = 50  # pages a reviewer can properly go through in a week
print(f"declining-with-demand candidates: {len(candidates):,} pages")
print(f"at {weekly_capacity} pages/week, covering all of them would take ~{len(candidates)/weekly_capacity:.0f} weeks")
print()
print("=> reviewing everything is not realistic; deciding the order is the whole problem.")


declining-with-demand candidates: 13,152 pages
at 50 pages/week, covering all of them would take ~263 weeks

=> reviewing everything is not realistic; deciding the order is the whole problem.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*
Three numbers from the starter dataset that make this lane worth the next seven weeks:

1. **13,152 of 30,000 pages (43.8%) are "declining with demand"** — trending down while still getting at least 100 impressions over 90 days. These aren't dead pages; they're losing ground while people are still searching for them.
2. **Declining pages hold 51.2% of all impressions in the dataset** (about 79.9M of 156M). Over half the visible demand sits on pages that are slipping — which is exactly the traffic a refresh queue is supposed to protect.
3. **9,328 pages dropped more than 30% in impressions in just the last 30 days** (51.8% of pages that had at least 100 impressions in the previous 30-day window). So the decay is recent and ongoing, not something that happened long ago.

One observation going the other way, for honesty: the "stale visible page" rule (no update in 180+ days plus 500+ impressions) catches only **17 pages** in this slice. Staleness alone is close to useless as a signal here — the real signal is in the trend and volume fields, which is where this lane focuses.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# The three numbers above, computed live
total_imp = df["impressions_90d"].sum()

declining = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]
print(f"1) declining with demand: {len(declining):,} / {len(df):,} pages ({len(declining)/len(df)*100:.1f}%)")

print(f"2) impressions on declining pages: {declining['impressions_90d'].sum():,} of {total_imp:,} ({declining['impressions_90d'].sum()/total_imp*100:.1f}%)")

prev = df[df["impressions_prev_30d"] >= 100]
dropped = prev[prev["impressions_last_30d"] < 0.7 * prev["impressions_prev_30d"]]
print(f"3) pages down >30% in the last 30 days: {len(dropped):,} ({len(dropped)/len(prev)*100:.1f}% of {len(prev):,})")

stale = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)]
print(f"   (for contrast: the 'stale visible' rule catches only {len(stale)} pages)")


1) declining with demand: 13,152 / 30,000 pages (43.8%)
2) impressions on declining pages: 79,887,612 of 156,010,989 (51.2%)
3) pages down >30% in the last 30 days: 9,328 (51.8% of 18,010)
   (for contrast: the 'stale visible' rule catches only 17 pages)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*
**What this work can say:**
- "Pages with these signals were *observed* to be more likely declining or worth review" — observational and directional claims, tied to the data.
- "The ranked queue is *decision-support*: it orders pages for a human to review; top-50 precision was X on held-out clients" — measured claims with a stated scope.
- "This page was flagged for reason R" — reason codes a reviewer can inspect and override.

**What it can't and won't claim:**
- That refreshing a page *causes* recovery. Proving that needs an experiment or a causal design, and this data can't provide one.
- Anything about Google's algorithm — I only see these clients' observed signals, not why rankings move.
- That `trend_direction == "down"` is ground truth. It's a current-window proxy; the plan is to replace it with a future-window label, and until then every result gets caveated as "by the proxy label."
- Anything client-identifiable. All outputs stay at pseudonymized IDs and aggregates.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Guardrail: confirm no product decision columns exist in the data that could leak
product_cols = {"health_score", "priority_score", "action_type", "needs_ctr_fix", "is_quick_win", "refresh_tier"}
present = product_cols.intersection(df.columns)
print("product decision columns in my data:", present if present else "none - safe")
print("(label and features must come from observable signals only)")


product decision columns in my data: none - safe
(label and features must come from observable signals only)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.